In [1]:
import gurobipy as gp
from gurobipy import GRB
import random
import itertools
import pandas as pd
import math
import numpy as np
from math import ceil
from InstanceSolver import InstanceSolver
from prepare_parameters import prepare_parameters
import pandas as pd

In [2]:
def parse_instance_file(filename):
    coords = {} # {i: (X, Y)}
    p = {}      # {i: prize}
    
    with open(filename, 'r') as f:
        lines = f.readlines()
        
    # 第一行是节点数 N
    N_size = int(lines[0].strip())
    N = list(range(N_size)) # 节点集合 N = {0, 1, ..., N-1}
    
    # 解析节点数据
    for i in N:
        parts = lines[i + 1].strip().split()
        X, Y, prize = float(parts[0]), float(parts[1]), float(parts[2])
        coords[i] = (X, Y)
        p[i] = prize
    return coords,p

In [3]:
L_ratio=0.25
N=10
i=2
alpha=1
beta=1


In [4]:
T_ratio=0.67
m=3
coords,p=parse_instance_file(f"poi-10-{i}.inst")
N=list(coords.keys())
para=prepare_parameters(coords,alpha,N,m,p,beta,L_ratio,T_ratio)
inst={"N":para.N,"A":para.A,"H":para.feasible_drone_times.keys(),"D":para.D,"p":para.p,"q":para.q,"t":para.truck_times,"tprime":para.feasible_drone_times,"T":para.T,"M":para.M}


Set parameter LicenseID to value 2698635


In [5]:
instance=InstanceSolver(inst, 1)
result=instance.solve()
result

TypeError: InstanceSolver.__init__() got an unexpected keyword argument 'test'

In [ ]:
L_ratio=0.25
N=10
i=2
alpha=1
beta=1
T_ratio=0.6666666667
m=3
coords,p=parse_instance_file(f"poi-10-{i}.inst")
N=list(coords.keys())
para=prepare_parameters(coords,alpha,N,m,p,beta,L_ratio,T_ratio)
inst={"N":para.N,"A":para.A,"H":para.feasible_drone_times.keys(),"D":para.D,"p":para.p,"q":para.q,"t":para.truck_times,"tprime":para.feasible_drone_times,"T":para.T,"M":para.M}
instance=InstanceSolver(inst)
result=instance.solve()

result

In [ ]:
if __name__=="__main__":
    L_ratios=[0.25,0.5,0.75,1,1.25]
    alphas=[1,2]
    betas=[0.67,1,1.33]
    T_ratios=[1/3,2/3]
    ms=[1,2,3]
    ns=[10,20]
    
    # 初始化结果字典，包含所有参数
    results={
        "L_ratio":[],
        "n":[],
        "instance_id":[],
        "alpha":[],
        "beta":[],
        "T_ratio":[],
        "m":[],
        "Runtime":[],
        "SolCount":[],
        "NodeCount":[],
        "status":[],
        "ObjVal":[]
    }
    
    cnt=1
    total_instances = len(L_ratios) * len(ns) * 5 * len(alphas) * len(betas) * len(T_ratios) * len(ms)
    print(f"Total instances to solve: {total_instances}")
    
    for L_ratio in L_ratios:
        for n in ns:
            for i in range(1,6):
                for alpha in alphas:
                    for beta in betas:
                        for T_ratio in T_ratios:
                            for m in ms:
                                try:
                                    coords,p=parse_instance_file(f"poi-{n}-{i}.inst")
                                    N=list(coords.keys())
                                    para=prepare_parameters(coords,alpha,N,m,p,beta,L_ratio,T_ratio)
                                    inst={
                                        "N":para.N,
                                        "A":para.A,
                                        "H":para.feasible_drone_times.keys(),
                                        "D":para.D,
                                        "p":para.p,
                                        "q":para.q,
                                        "t":para.truck_times,
                                        "tprime":para.feasible_drone_times,
                                        "T":para.T,
                                        "M":para.M
                                    }
                                    instance=InstanceSolver(inst)
                                    result=instance.solve()
                                    
                                    # 记录所有参数和结果
                                    results["L_ratio"].append(L_ratio)
                                    results["n"].append(n)
                                    results["instance_id"].append(i)
                                    results["alpha"].append(alpha)
                                    results["beta"].append(beta)
                                    results["T_ratio"].append(T_ratio)
                                    results["m"].append(m)
                                    results["status"].append(result["status"])
                                    
                                    if result["status"]==2:
                                        results["Runtime"].append(result["Runtime"])
                                        results["SolCount"].append(result["SolCount"])
                                        results["NodeCount"].append(result["NodeCount"])
                                        results["ObjVal"].append(result["ObjVal"])
                                    else:
                                        results["Runtime"].append(None)
                                        results["SolCount"].append(None)
                                        results["NodeCount"].append(None)
                                        print(f"Warning: Instance failed - i={i}, alpha={alpha}, beta={beta}, T_ratio={T_ratio}, m={m}")
                                        print(f"Status: {result}")
                                    
                                    if cnt%10==0:
                                        print(f"{cnt}/{total_instances} instances finished ({cnt/total_instances*100:.1f}%)")
                                    cnt+=1
                                    
                                except Exception as e:
                                    print(f"Error at instance {cnt}: {e}")
                                    print(f"Parameters: n={n}, i={i}, alpha={alpha}, beta={beta}, T_ratio={T_ratio}, m={m}")
                                    cnt+=1
                                    continue
    
    # 保存最终结果
    df = pd.DataFrame(results)
    
    # 保存到Excel
    output_file = f'obj_results_allinstance.xlsx'

In [ ]:
df.to_excel(output_file, index=False)